## 03 — Gold Layer: Star Schema
Builds the dimensional model consumed by Power BI.

**Schema design decisions:**
- `dim_category` merged into `dim_product` — `category_name` and `category_description` inline
- `dim_shipper` merged into `fact_sales` — `shipper_name` and `shipper_phone` inline
- `dim_region` merged into `dim_territory` — `region_description` inline

In [ ]:
# pyright: reportMissingImports=false, reportUndefinedVariable=false
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, year, quarter, month, dayofmonth, dayofweek,
    date_format, concat, lpad, sequence, explode,
)

### Configuration

In [ ]:
catalog = "workspace"
bronze_schema = "indicium_bronze"
silver_schema = "indicium_silver"
gold_schema = "indicium_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")
print(f"Schema ready: {catalog}.{gold_schema}")

### Drop Existing Gold Tables

In [ ]:
tables_to_drop = [
    "dim_product",
    "dim_category",
    "dim_customer",
    "dim_employee",
    "dim_shipper",
    "dim_territory",
    "dim_region",
    "dim_calendar",
    "bridge_employee_territory",
    "fact_sales",
]

for table in tables_to_drop:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{gold_schema}.{table}")
    print(f"Dropped (if existed): {catalog}.{gold_schema}.{table}")

print("\nCleanup complete.")

### `dim_product`
Products enriched with category information (category merged inline).

In [ ]:
products = spark.table(f"{catalog}.{bronze_schema}.products")
categories = spark.table(f"{catalog}.{bronze_schema}.categories")

dim_product = (
    products.select(
        col("product_id").cast("int").alias("product_id"),
        col("product_name"),
        col("supplier_id").cast("int").alias("supplier_id"),
        col("category_id").cast("int").alias("category_id"),
        col("quantity_per_unit"),
        col("unit_price").cast("double").alias("list_price"),
        col("units_in_stock").cast("int").alias("units_in_stock"),
        col("units_on_order").cast("int").alias("units_on_order"),
        col("reorder_level").cast("int").alias("reorder_level"),
        col("discontinued").cast("int").alias("discontinued"),
    )
    .join(
        categories.select(
            col("category_id").cast("int").alias("category_id"),
            col("category_name"),
            col("description").alias("category_description"),
        ),
        on="category_id",
        how="left",
    )
)

dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.dim_product"
)
print(f"dim_product: {spark.table(f"{catalog}.{gold_schema}.dim_product").count():,} rows")

### `dim_customer`

In [ ]:
customers = spark.table(f"{catalog}.{bronze_schema}.customers")

dim_customer = customers.select(
    col("customer_id"),
    col("company_name"),
    col("contact_name"),
    col("contact_title"),
    col("city"),
    col("region"),
    col("country"),
)

dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.dim_customer"
)
print(f"dim_customer: {spark.table(f"{catalog}.{gold_schema}.dim_customer").count():,} rows")

### `dim_employee`

In [ ]:
employees = spark.table(f"{catalog}.{bronze_schema}.employees")

dim_employee = employees.select(
    col("employee_id").cast("int").alias("employee_id"),
    col("first_name"),
    col("last_name"),
    col("title"),
    col("city"),
    col("region"),
    col("country"),
)

dim_employee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.dim_employee"
)
print(f"dim_employee: {spark.table(f"{catalog}.{gold_schema}.dim_employee").count():,} rows")

### `dim_territory`
Territories enriched with region description (region merged inline).

In [ ]:
territories = spark.table(f"{catalog}.{bronze_schema}.territories")
region = spark.table(f"{catalog}.{bronze_schema}.region")

dim_territory = (
    territories.select(
        col("territory_id"),
        col("territory_description"),
        col("region_id").cast("int").alias("region_id"),
    )
    .join(
        region.select(
            col("region_id").cast("int").alias("region_id"),
            col("region_description"),
        ),
        on="region_id",
        how="left",
    )
)

dim_territory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.dim_territory"
)
print(f"dim_territory: {spark.table(f"{catalog}.{gold_schema}.dim_territory").count():,} rows")

### `dim_calendar`
Generated date spine covering the full range of order dates in the dataset.

In [ ]:
date_range = (
    spark.table(f"{catalog}.{silver_schema}.orders_clean")
    .select(F.min("order_date").alias("min_date"), F.max("order_date").alias("max_date"))
    .collect()[0]
)

dim_calendar = (
    spark.range(1)
    .select(explode(sequence(lit(date_range["min_date"]), lit(date_range["max_date"]))).alias("date"))
    .withColumn("date_key", (year("date") * 10000 + month("date") * 100 + dayofmonth("date")).cast("int"))
    .withColumn("year", year("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("is_weekday", dayofweek("date").isin([2, 3, 4, 5, 6]))
    .withColumn(
        "year_month",
        concat(year("date").cast("string"), lit("-"), lpad(month("date").cast("string"), 2, "0")),
    )
)

dim_calendar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.dim_calendar"
)
print(f"dim_calendar: {spark.table(f"{catalog}.{gold_schema}.dim_calendar").count():,} rows")

### `bridge_employee_territory`

In [ ]:
emp_territories = spark.table(f"{catalog}.{bronze_schema}.employee_territories")

bridge_employee_territory = emp_territories.select(
    col("employee_id").cast("int").alias("employee_id"),
    col("territory_id"),
)

bridge_employee_territory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.bridge_employee_territory"
)
print(f"bridge_employee_territory: {spark.table(f"{catalog}.{gold_schema}.bridge_employee_territory").count():,} rows")

### `fact_sales`
Sales line items with shipper and employee attributes denormalized inline.

In [ ]:
sales_line = spark.table(f"{catalog}.{silver_schema}.sales_line")

emp_attrs = (
    spark.table(f"{catalog}.{bronze_schema}.employees")
    .select(
        col("employee_id").cast("int").alias("employee_id"),
        col("first_name").alias("employee_first_name"),
        col("last_name").alias("employee_last_name"),
        col("title").alias("employee_title"),
    )
)

ship_attrs = (
    spark.table(f"{catalog}.{bronze_schema}.shippers")
    .select(
        col("shipper_id").cast("int").alias("shipper_id"),
        col("company_name").alias("shipper_name"),
        col("phone").alias("shipper_phone"),
    )
)

fact_sales = (
    sales_line
    .withColumn(
        "order_date_key",
        (year("order_date") * 10000 + month("order_date") * 100 + dayofmonth("order_date")).cast("int"),
    )
    .withColumnRenamed("ship_via", "shipper_id")
    .join(emp_attrs, on="employee_id", how="left")
    .join(ship_attrs, on="shipper_id", how="left")
    .select(
        col("order_id"),
        col("product_id"),
        col("customer_id"),
        col("employee_id"),
        col("shipper_id"),
        col("order_date"),
        col("order_date_key"),
        col("required_date"),
        col("shipped_date"),
        col("ship_country"),
        col("ship_region"),
        col("ship_city"),
        col("unit_price"),
        col("quantity"),
        col("discount"),
        col("gross_sales"),
        col("discount_value"),
        col("net_sales"),
        col("shipped_on_time_flag"),
        col("shipping_delay_days"),
        col("employee_first_name"),
        col("employee_last_name"),
        col("employee_title"),
        col("shipper_name"),
        col("shipper_phone"),
    )
)

fact_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.fact_sales"
)
print(f"fact_sales: {spark.table(f"{catalog}.{gold_schema}.fact_sales").count():,} rows")

### Validation

In [ ]:
gold_tables = [
    "dim_product",
    "dim_customer",
    "dim_employee",
    "dim_territory",
    "dim_calendar",
    "bridge_employee_territory",
    "fact_sales",
]

print("Gold layer — row counts:")
for table in gold_tables:
    count = spark.table(f"{catalog}.{gold_schema}.{table}").count()
    print(f"  {table}: {count:,}")